# Savant 07 evaluate the self-play adapters

This notebook chooses a checkpoint by task results and then measures it once on the bench. Each adapter from 06-train is merged into the base model and served with vLLM on both GPUs, with greedy decoding.

1. The base model and every checkpoint answer 30 development tables with seeds from 4,500,000, which training never saw. The best checkpoint is the one with no false pairs, the highest share of confirmed pairs named, then the fewest failed code steps.
2. The base model and the chosen checkpoint then answer the 30 guided bench tables used before and 40 bench tables that no model has seen, in the same session.


In [ ]:
import glob
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

build = sorted(glob.glob("/kaggle/input/**/BUILD", recursive=True))
assert build and Path(build[0]).read_text().strip() == "EXPECTED_BUILD", "stale bench dataset, stop"
wheel = sorted(glob.glob("/kaggle/input/**/savant-*.whl", recursive=True))[-1]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", wheel, "vllm", "peft"], check=True)
ADAPTERS = sorted(
    {
        str(Path(p).parent)
        for p in glob.glob("/kaggle/input/**/adapters/*/adapter_config.json", recursive=True)
    }
)
TASKS = sorted(glob.glob("/kaggle/input/**/tasks/*.csv", recursive=True))
print(len(ADAPTERS), "adapters", len(TASKS), "bench tables")
assert ADAPTERS, "no adapters from 06-train"

In [ ]:
# Settings for this run.
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
DEV_TABLES = 30
DEV_SEED = 4_500_000  # training used seeds from 4,000,000 to 4,000,299
SEEN = {"null": (0, 5), "hard-null": (0, 5), "plant": (0, 20)}  # the 30 guided tables used so far
UNSEEN = {"null": (5, 15), "hard-null": (5, 15), "plant": (20, 40)}  # never shown to any model
WORKERS = 32
KEEP_MESSAGES = 6
MAX_NEW_TOKENS = 768
MERGED = Path("/tmp/merged")

In [ ]:
PORTS = [8000, 8001]


def start(path, gpu, port, extra=()):
    log = open(f"/kaggle/working/vllm-{gpu}.log", "w")  # noqa: SIM115, the server keeps it
    cmd = [
        "vllm", "serve", str(path),
        "--served-model-name", "savant",
        "--dtype", "half",
        "--max-model-len", "8192",
        "--gpu-memory-utilization", "0.90",
        "--port", str(port),
        *extra,
    ]  # fmt: skip
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
    return subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)


def healthy(port, procs, limit=900):
    end = time.time() + limit
    while time.time() < end:
        if any(p.poll() is not None for p in procs):
            return False
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=5)
            return True
        except Exception:
            time.sleep(10)
    return False


EXTRA = []


def serve(path):
    global EXTRA
    procs = [start(path, g, p, EXTRA) for g, p in enumerate(PORTS)]
    if all(healthy(p, procs) for p in PORTS):
        return procs
    for p in procs:
        p.kill()
    time.sleep(10)
    EXTRA = ["--enforce-eager"]
    procs = [start(path, g, p, EXTRA) for g, p in enumerate(PORTS)]
    if all(healthy(p, procs) for p in PORTS):
        return procs
    print(Path("/kaggle/working/vllm-0.log").read_text()[-4000:])
    raise SystemExit("vLLM did not start")


def stop(procs):
    for p in procs:
        p.kill()
    for p in procs:
        p.wait()
    time.sleep(5)


def merge(adapter):
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer

    out = MERGED / Path(adapter).name
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16)
    PeftModel.from_pretrained(base, adapter).merge_and_unload().save_pretrained(out)
    AutoTokenizer.from_pretrained(MODEL_ID).save_pretrained(out)
    return out

In [ ]:
import itertools
import threading

import polars as pl

from savant.llm import OpenAICompatible
from savant.planted import Plant, Task
from savant.selfplay import collect, dev_tasks
from savant.traces import PAIR_QUESTION

lock = threading.Lock()
ports = itertools.cycle(PORTS)


def client():
    with lock:
        port = next(ports)
    return OpenAICompatible(
        f"http://127.0.0.1:{port}/v1", "savant", temperature=0.0, max_tokens=MAX_NEW_TOKENS
    )


def load_task(csv):
    truth = json.loads(Path(csv.replace(".csv", ".truth.json")).read_text())
    plants = [Plant(p["kind"], tuple(p["columns"]), p["strength"]) for p in truth["plants"]]
    return Task(truth["id"], pl.read_csv(csv, try_parse_dates=True), plants)


bench = [load_task(c) for c in TASKS]


def pick(spans):
    out = []
    for prefix, (lo, hi) in spans.items():
        out += [
            t for t in bench if t.id.startswith(prefix + "-") and t.id[len(prefix) + 1 :].isdigit()
        ][lo:hi]
    return [(t, PAIR_QUESTION) for t in out]


log = Path("/kaggle/working/runs.jsonl")
log.write_text("")


def measure(label, part, tasks):
    def on_record(r):
        row = {k: v for k, v in r.items() if k not in ("messages", "learn")}
        row.update(model=label, part=part, transcript=r["messages"])
        with lock, log.open("a") as f:
            f.write(json.dumps(row, default=str) + "\n")

    t0 = time.time()
    recs = collect(
        tasks, client, samples=1, workers=WORKERS, keep_messages=KEEP_MESSAGES,
        scan_workers=os.cpu_count() or 2, on_record=on_record,
    )  # fmt: skip
    return score(recs, time.time() - t0)


def score(recs, seconds):
    planted = [r for r in recs if r["kind"] == "plant"]

    def per(kind):
        rs = [r for r in recs if r["kind"] == kind]
        return round(sum(r["false"] for r in rs) / len(rs), 3) if rs else None

    confirmed = sum(r["confirmed"] for r in planted)
    return {
        "tables": len(recs),
        "grounded_answers": round(sum(r["grounded"] for r in recs) / len(recs), 3),
        "mean_rounds": round(sum(r["rounds"] for r in recs) / len(recs), 2),
        "failed_step_share": round(
            sum(r["failed"] for r in recs) / max(1, sum(r["steps"] for r in recs)), 3
        ),
        "planted_pairs": sum(r["planted"] for r in planted),
        "recall": round(
            sum(r["found"] for r in planted) / max(1, sum(r["planted"] for r in planted)), 3
        ),
        "confirmed_pairs": confirmed,
        "confirmed_named": round(
            sum(r["fidelity"] * r["confirmed"] for r in planted) / max(1, confirmed), 3
        ),
        "false_per_null_table": per("null"),
        "false_per_hard_null_table": per("hard-null"),
        "false_per_planted_table": per("plant"),
        "errors": sum(bool(r["error"]) for r in recs),
        "seconds": round(seconds),
    }

In [ ]:
dev = dev_tasks(DEV_TABLES, DEV_SEED)
models = [("base", MODEL_ID)]
results = {"dev": {}, "bench": {}}
for adapter in ADAPTERS:
    models.append((Path(adapter).name, None))
for label, path in models:
    if path is None:
        path = merge(next(a for a in ADAPTERS if Path(a).name == label))
    procs = serve(path)
    results["dev"][label] = measure(label, "dev", dev)
    print(label, json.dumps(results["dev"][label]), flush=True)
    stop(procs)
    if path != MODEL_ID:
        shutil.rmtree(path, ignore_errors=True)


def rank(label):
    d = results["dev"][label]
    false = sum(
        d[k] or 0
        for k in ("false_per_null_table", "false_per_hard_null_table", "false_per_planted_table")
    )
    return (false > 0, -d["confirmed_named"], d["failed_step_share"], d["mean_rounds"])


best = min((label for label, _ in models if label != "base"), key=rank)
results["chosen"] = best
print("chosen checkpoint", best)

In [ ]:
for label in ["base", best]:
    path = (
        MODEL_ID if label == "base" else merge(next(a for a in ADAPTERS if Path(a).name == label))
    )
    procs = serve(path)
    results["bench"][label] = {
        "seen": measure(label, "seen", pick(SEEN)),
        "unseen": measure(label, "unseen", pick(UNSEEN)),
    }
    print(label, json.dumps(results["bench"][label]), flush=True)
    stop(procs)
results["vllm_extra"] = EXTRA
Path("/kaggle/working/eval.json").write_text(json.dumps(results, indent=1))
print(json.dumps(results, indent=1))